# Module 3: Vision Transformers in Keras

This notebook loads a CNN, uses one of its feature layers, builds a CNN-Vision Transformer hybrid, compiles it, and defines its training configuration.

## Setup

Import TensorFlow and locate the saved CNN and image dataset.

In [ ]:
import os
import random
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, Model
from tensorflow.keras.preprocessing.image import ImageDataGenerator

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

CNN_PATH = './best_model.keras'
if not os.path.exists(CNN_PATH):
    CNN_PATH = os.path.join('.', 'ai-capstone-keras-best-model-model_downloaded.keras')
DATASET_PATH = './images_dataSAT'
if not os.path.exists(DATASET_PATH):
    DATASET_PATH = os.path.join('.', 'AI Capstone DL Projects', 'CNN Model Development', 'images_dataSAT')
print('CNN path:', CNN_PATH)
print('CNN exists:', os.path.exists(CNN_PATH))
print('Dataset path:', DATASET_PATH)
print('Dataset exists:', os.path.exists(DATASET_PATH))

## Task 1: Load the pre-trained CNN model

Load the model into `cnn_model` with `load_model()` and print its summary.

In [ ]:
cnn_model = tf.keras.models.load_model(CNN_PATH, compile=False)
cnn_model.trainable = False

# Calling a loaded Sequential model creates its symbolic input tensor.
input_shape = tuple(cnn_model.input_shape[1:])
cnn_model(tf.zeros((1,) + input_shape))
cnn_model.summary()

## Task 2: Select the CNN feature layer

Select the last four-dimensional layer, which provides a spatial feature map for token creation.

In [ ]:
feature_layer = None
for layer in reversed(cnn_model.layers):
    if hasattr(layer, 'output') and len(layer.output.shape) == 4:
        feature_layer = layer
        break
if feature_layer is None:
    raise ValueError('No 4D feature layer was found.')
feature_layer_name = feature_layer.name
print('feature_layer_name:', feature_layer_name)
print('Feature shape:', feature_layer.output.shape)

## Hybrid architecture

The function below reshapes CNN feature maps into tokens, adds position information, applies transformer blocks, and creates a classifier.

In [ ]:
class AddPositionEmbedding(layers.Layer):
    def __init__(self, token_count, embed_dim, **kwargs):
        super().__init__(**kwargs)
        self.position_embedding = self.add_weight(name='position_embedding', shape=(1, token_count, embed_dim), initializer='random_normal')
    def call(self, tokens):
        return tokens + self.position_embedding

class TransformerBlock(layers.Layer):
    def __init__(self, embed_dim, num_heads=4, mlp_dim=128, dropout=0.1, **kwargs):
        super().__init__(**kwargs)
        self.attention = layers.MultiHeadAttention(num_heads=num_heads, key_dim=embed_dim)
        self.mlp = tf.keras.Sequential([layers.Dense(mlp_dim, activation='gelu'), layers.Dropout(dropout), layers.Dense(embed_dim), layers.Dropout(dropout)])
        self.norm1 = layers.LayerNormalization(epsilon=1e-6)
        self.norm2 = layers.LayerNormalization(epsilon=1e-6)
    def call(self, tokens, training=None):
        attended = self.attention(tokens, tokens, training=training)
        tokens = self.norm1(tokens + attended)
        return self.norm2(tokens + self.mlp(tokens, training=training))

def build_cnn_vit_hybrid(cnn_model, feature_layer_name, num_classes=2):
    feature_output = cnn_model.get_layer(feature_layer_name).output
    height, width, channels = feature_output.shape[1:]
    tokens = layers.Reshape((height * width, channels))(feature_output)
    tokens = AddPositionEmbedding(height * width, channels)(tokens)
    tokens = TransformerBlock(channels, mlp_dim=max(64, channels * 2))(tokens)
    tokens = TransformerBlock(channels, mlp_dim=max(64, channels * 2))(tokens)
    outputs = layers.Dense(num_classes, activation='softmax')(layers.GlobalAveragePooling1D()(tokens))
    return Model(cnn_model.input, outputs, name='cnn_vit_hybrid')

## Task 3: Define `hybrid_model`

Build the hybrid architecture with `build_cnn_vit_hybrid`.

In [ ]:
hybrid_model = build_cnn_vit_hybrid(cnn_model, feature_layer_name, num_classes=2)
hybrid_model.summary()

## Task 4: Compile `hybrid_model`

Use Adam, categorical cross-entropy, and accuracy.

In [ ]:
hybrid_model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4), loss='categorical_crossentropy', metrics=['accuracy'])
print('hybrid_model compiled successfully')

## Task 5: Define the training configuration

Create the generators, checkpoint callback, and training settings.

In [ ]:
image_size = (64, 64)
batch_size = 8
epochs = 3
data_generator = ImageDataGenerator(rescale=1.0 / 255.0, validation_split=0.2, rotation_range=20, horizontal_flip=True)
train_generator = data_generator.flow_from_directory(DATASET_PATH, target_size=image_size, batch_size=batch_size, class_mode='categorical', subset='training', shuffle=True, seed=SEED)
validation_generator = data_generator.flow_from_directory(DATASET_PATH, target_size=image_size, batch_size=batch_size, class_mode='categorical', subset='validation', shuffle=False, seed=SEED)
checkpoint_callback = tf.keras.callbacks.ModelCheckpoint('best_cnn_vit_hybrid.keras', monitor='val_accuracy', mode='max', save_best_only=True)
training_config = {'epochs': epochs, 'steps_per_epoch': len(train_generator), 'validation_steps': len(validation_generator), 'callbacks': [checkpoint_callback]}
print({key: value for key, value in training_config.items() if key != 'callbacks'})
print('Training configuration defined successfully')

## Completed

All five Module 3 tasks are complete and ready to run.